# 09 · Feature engineering and pipelines

After this notebook you can choose a scaler, encode categorical columns, impute missing values, put numeric, categorical and text features into one `ColumnTransformer` + `Pipeline`, prove why preprocessing must live *inside* cross-validation, save and reload the whole thing with `joblib`, and ship an end-to-end churn model with a prediction function.

**Time:** ~35 min · **Runs:** offline on CPU in < 1 min · **Needs:** [01 · ML workflow](01_ml_workflow_and_train_test_split.ipynb), [05 · Cross-validation](05_overfitting_regularization_cross_validation.ipynb)

## Why this matters in interviews

- Real data is a DataFrame of mixed columns: numbers with outliers, categories with typos, free text, missing values. "Walk me through your preprocessing" is a standard practical question, and so is the take-home that tests it ([ML model end to end](../17_interview_coding_rounds/06_ml_model_end_to_end.ipynb)).
- A `Pipeline` is the answer to three follow-ups at once: how do you avoid leakage in CV, how do you guarantee training and serving use the same transformations (training/serving skew), and what exactly did you deploy?
- The same discipline applies to LLM systems: the prompt template, retrieval config and model version are the "preprocessing", and they must ship and be versioned together.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `dt20` | What is a feature store and does an LLM system need one? |
| `pd23` | How do you version an AI system so results are reproducible? |
| `ev17` | How do you evaluate a classification task done by an LLM? |

**Also asked in classical-ML rounds:** StandardScaler vs MinMaxScaler vs RobustScaler; one-hot vs ordinal vs target encoding; how do you handle missing values, and is missingness a signal? how do you handle an unseen category at inference? why must feature selection happen inside CV? how do you deploy a scikit-learn model?

In [ ]:
import tempfile
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, OrdinalEncoder, RobustScaler, StandardScaler

rng = np.random.default_rng(0)
BLUE, ORANGE, AQUA, YELLOW, GREY = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#8a8984"
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.prop_cycle": plt.cycler(color=[BLUE, ORANGE, AQUA, YELLOW, "#4a3aa7"])})

## 1. Scaling: which scaler, when

**In plain English:** many models compare features by size (distances, dot products, penalties), so a column in thousands swamps a column in single digits. Scaling puts columns on a comparable footing. Which scaler depends on outliers.

A monthly-spend column where a few customers spend 20× the rest:

In [ ]:
spend = np.r_[rng.normal(60, 15, 495), [900, 1100, 1200, 1500, 2000]].reshape(-1, 1)   # 5 extreme customers
scaled = {"StandardScaler": StandardScaler().fit_transform(spend),
          "MinMaxScaler": MinMaxScaler().fit_transform(spend),
          "RobustScaler": RobustScaler().fit_transform(spend)}
fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))
for ax, (name, v) in zip(axes, scaled.items()):
    typical = v[:495, 0]
    ax.hist(typical, bins=30, color=BLUE)
    ax.set(title=f"{name}\ntypical customers span {typical.min():.2f} to {typical.max():.2f}", xlabel="scaled value")
axes[0].set_ylabel("customers"); plt.tight_layout(); plt.show()
for name, v in scaled.items():
    print(f"{name:>15}: 495 typical customers squeezed into a range of width {np.ptp(v[:495]):.3f}; outliers reach {v.max():.1f}")
assert np.ptp(scaled["RobustScaler"][:495]) > np.ptp(scaled["StandardScaler"][:495]) > np.ptp(scaled["MinMaxScaler"][:495])

| Scaler | Formula | Outliers | Use when |
|---|---|---|---|
| `StandardScaler` | (x − mean) / std | pull the mean and inflate the std | the default for linear models, SVMs, PCA, kNN |
| `MinMaxScaler` | (x − min) / (max − min) → [0, 1] | one extreme squashes everyone else | bounded features (pixels, percentages), neural-net inputs |
| `RobustScaler` | (x − median) / IQR | ignored (median and quartiles) | heavy outliers you want to keep |
| `np.log1p` / `PowerTransformer` | reshape the distribution | tames long right tails | money, counts, durations |
| none | | | tree models (order is all that matters) |

## 2. Encoding categories

| Encoder | Output | Use when | Trap |
|---|---|---|---|
| `OneHotEncoder` | one 0/1 column per category | nominal categories (region, browser) for linear models | high cardinality explodes the width; use `handle_unknown="ignore"` |
| `OrdinalEncoder` | one integer column | *ordered* categories (basic < pro < enterprise), and trees | on nominal data a linear model reads a fake order |
| `TargetEncoder` | mean target per category (cross-fitted) | high-cardinality columns (zip code, product id) | done outside CV it leaks the label |

In [ ]:
plans = pd.DataFrame({"plan": ["basic", "enterprise", "pro", "basic"], "region": ["north", "south", "east", "north"]})
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False).fit(plans[["region"]])
print("one-hot columns:", ohe.get_feature_names_out().tolist())
print("unseen region 'mars' ->", ohe.transform(pd.DataFrame({"region": ["mars"]})).tolist(), "(all zeros, no crash)")

ord_enc = OrdinalEncoder(categories=[["basic", "pro", "enterprise"]]).fit(plans[["plan"]])  # state the order explicitly
print("ordinal codes:", {k: float(v) for k, v in zip(plans["plan"], ord_enc.transform(plans[["plan"]]).ravel())})
assert ord_enc.transform(pd.DataFrame({"plan": ["enterprise"]}))[0, 0] > ord_enc.transform(pd.DataFrame({"plan": ["basic"]}))[0, 0]

Always pass `categories=` to `OrdinalEncoder`; the default is alphabetical, which would rank "enterprise" below "pro".

## 3. Missing values

`SimpleImputer` fills gaps with the median, mean, most frequent value or a constant. It is fitted on training data like any transformer. `add_indicator=True` also adds a 0/1 "was missing" column, because **missingness is often a signal** (a customer who never filled in their phone number behaves differently).

In [ ]:
income = pd.DataFrame({"income": [52_000, np.nan, 61_000, np.nan, 48_000, 250_000]})
imp = SimpleImputer(strategy="median", add_indicator=True).set_output(transform="pandas").fit(income)
print(imp.transform(income))
assert imp.statistics_[0] == np.median([52_000, 61_000, 48_000, 250_000])

The median ignores the 250k outlier; the mean would not. For trees, `HistGradientBoosting*` handles NaN natively, so you can often skip imputation there.

## 4. `ColumnTransformer` + `Pipeline`: one object for all of it

`ColumnTransformer` sends each group of columns through its own steps and glues the results side by side. Wrap it in a `Pipeline` with the model and you get one estimator with `fit` / `predict`, which can be cross-validated, grid-searched, saved and served as a unit.

```
                     ┌─ numeric:     impute(median, +indicator) → StandardScaler ─┐
raw DataFrame  ──►   ├─ categorical: impute("missing") → OneHotEncoder ────────────┼──►  LogisticRegression  ──►  P(churn)
                     └─ text:        TfidfVectorizer(1-2 grams) ───────────────────┘
```

First the data: a generated customer-churn table with numeric, categorical and free-text columns, and churn driven by all three.

In [ ]:
def make_churn_data(n=2000, seed=1):
    r = np.random.default_rng(seed)
    unhappy = ["thinking of cancelling my subscription", "this is too expensive for what we get",
               "the app is slow and keeps crashing", "we are switching to a competitor next month",
               "support never answers, I want to cancel", "price went up again, not worth it"]
    neutral = ["how do I export my data to csv", "thanks for the quick help yesterday",
               "please update my billing address", "love the new dashboard",
               "can I add another user to the account", "what are your opening hours"]
    is_unhappy = r.random(n) < 0.25
    text = np.where(is_unhappy, r.choice(unhappy, n), r.choice(neutral, n))
    df = pd.DataFrame({
        "tenure_months": r.integers(1, 72, n),
        "monthly_charges": r.normal(70, 20, n).clip(20, 150).round(2),
        "support_calls": r.poisson(1.5, n),
        "plan": r.choice(["basic", "pro", "enterprise"], n, p=[0.5, 0.35, 0.15]),
        "contract": r.choice(["monthly", "annual"], n, p=[0.6, 0.4]),
        "region": r.choice(["north", "south", "east", "west"], n),
        "last_ticket": text,
    })
    logit = (-1.2 - 0.04 * df.tenure_months + 0.02 * (df.monthly_charges - 70) + 0.3 * df.support_calls
             + 1.0 * (df.contract == "monthly") - 0.6 * (df.plan == "enterprise") + 1.8 * is_unhappy)
    df["churned"] = (r.random(n) < 1 / (1 + np.exp(-logit))).astype(int)
    df.loc[r.random(n) < 0.08, "monthly_charges"] = np.nan          # 8% missing
    df.loc[r.random(n) < 0.03, "plan"] = np.nan                     # 3% missing
    return df

churn = make_churn_data()
print(f"{len(churn)} customers, churn rate {churn.churned.mean():.1%}; missing values: {churn.isna().sum()[churn.isna().sum() > 0].to_dict()}")
churn.head()

In [ ]:
NUMERIC = ["tenure_months", "monthly_charges", "support_calls"]
CATEGORICAL = ["plan", "contract", "region"]
TEXT = "last_ticket"                                  # a single string, not a list: TF-IDF wants a 1-D column

def build_pipeline(numeric=NUMERIC, categorical=CATEGORICAL, text=TEXT, C=1.0):
    parts = [("num", make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler()), numeric)]
    if categorical:
        parts.append(("cat", make_pipeline(SimpleImputer(strategy="constant", fill_value="missing"),
                                           OneHotEncoder(handle_unknown="ignore")), categorical))
    if text:
        parts.append(("text", TfidfVectorizer(ngram_range=(1, 2), min_df=2), text))
    return Pipeline([("prep", ColumnTransformer(parts)), ("model", LogisticRegression(C=C, max_iter=2000))])

X, y = churn.drop(columns="churned"), churn["churned"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
pipe = build_pipeline().fit(X_tr, y_tr)
names = pipe.named_steps["prep"].get_feature_names_out()
print(f"{len(names)} model features from {X.shape[1]} raw columns, e.g.:", names[:4].tolist(), "...", names[-3:].tolist())
print(f"test accuracy {pipe.score(X_te, y_te):.3f}")

`get_feature_names_out()` tells you exactly which engineered column is which (`num__`, `cat__`, `text__` prefixes), which you need for reading coefficients or debugging. To see intermediate results as a DataFrame, call `.set_output(transform="pandas")` on a transformer; one-hot encoding then needs `sparse_output=False`, and TF-IDF (always sparse) cannot be shown that way.

### Feature selection, briefly

With thousands of TF-IDF or one-hot columns you can drop weak ones: `SelectKBest(f_classif, k=...)` keeps the columns with the strongest univariate relationship to the label; `SelectFromModel(LogisticRegression with L1)` keeps those an L1 model does not zero out ([notebook 05](05_overfitting_regularization_cross_validation.ipynb)). Selection is **fitted on labels**, so it must live inside the pipeline. The next section shows what happens when it does not.

## 5. Leakage-safe cross-validation: the wrong way vs the right way

The classic trap: 100 rows, 5,000 columns of **pure noise**, random labels. Nothing can be learned; honest accuracy is 50%. The "wrong way" picks the 20 columns most correlated with the label using *all* rows, then cross-validates. The right way puts the selection inside the pipeline, so each fold selects using its own training rows only.

In [ ]:
X_noise = rng.normal(size=(100, 5000))
y_noise = rng.integers(0, 2, size=100)
cv = StratifiedKFold(5, shuffle=True, random_state=0)

X_selected = SelectKBest(f_classif, k=20).fit_transform(X_noise, y_noise)      # WRONG: saw every label
wrong = cross_val_score(LogisticRegression(), X_selected, y_noise, cv=cv)
right = cross_val_score(make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression()), X_noise, y_noise, cv=cv)
print(f"wrong way (select, then CV): accuracy {wrong.mean():.2f}  <- looks like a real model")
print(f"right way (select inside CV): accuracy {right.mean():.2f}  <- the truth: coin flipping")
assert wrong.mean() > 0.75 and abs(right.mean() - 0.5) < 0.15

fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(["selection outside CV\n(leaky)", "selection inside the pipeline\n(honest)"], [wrong.mean(), right.mean()],
       color=[ORANGE, BLUE], yerr=[wrong.std(), right.std()], capsize=6)
ax.axhline(0.5, color=GREY, ls="--", label="chance")
ax.set(title="Pure-noise data: same model, same folds", ylabel="5-fold CV accuracy", ylim=(0, 1)); ax.legend(); plt.show()

The leaky score is pure fiction: the selector already "knew" which noise columns happened to line up with the labels, including the labels of every validation fold. The same trap applies to target encoding, oversampling (SMOTE) before splitting, fitting a TF-IDF vocabulary or a scaler on all rows, and deduplicating after splitting. The rule: **every step that is fitted goes inside the pipeline**, and the pipeline goes inside the CV.

The same idea at system scale is the **feature store** (`dt20`): one definition of each feature, computed identically for training and serving, with point-in-time correctness so training rows never see the future. A pipeline object is the single-model version of that promise.

## 6. Saving and loading with `joblib`

Save the **fitted pipeline**, not just the model: the imputation medians, the scaler's statistics, the one-hot vocabulary and the TF-IDF vocabulary are all part of what you deploy. Record the library versions next to it (`pd23`): a pickle is only guaranteed to load with the same scikit-learn version.

In [ ]:
tmp = Path(tempfile.mkdtemp())                                  # scratch space, never the repo
model_path = tmp / "churn_pipeline.joblib"
artifact = {"pipeline": pipe, "sklearn_version": sklearn.__version__, "features": list(X.columns),
            "trained_rows": len(X_tr), "test_accuracy": round(pipe.score(X_te, y_te), 4)}
joblib.dump(artifact, model_path)
loaded = joblib.load(model_path)
print(f"saved {model_path.name} ({model_path.stat().st_size / 1024:.0f} KB), trained with scikit-learn {loaded['sklearn_version']}")
assert np.allclose(loaded["pipeline"].predict_proba(X_te), pipe.predict_proba(X_te))   # identical predictions
assert loaded["sklearn_version"] == sklearn.__version__

Only load pickles you created or trust: unpickling can execute arbitrary code. For cross-language serving, export to ONNX (`skl2onnx`, not installed here) or wrap the loaded pipeline in an API ([FastAPI serving](../16_production/04_fastapi_serving_and_streaming.ipynb)).

## 7. End to end: which features earn their place?

Cross-validate three pipelines on the training set: numeric columns only, numeric + categorical, and everything including the ticket text. Then refit the best on all training rows and score the test set once.

In [ ]:
candidates = {
    "numeric only": build_pipeline(categorical=None, text=None),
    "+ categorical": build_pipeline(text=None),
    "+ categorical + ticket text": build_pipeline(),
}
cv5 = StratifiedKFold(5, shuffle=True, random_state=0)
results = {name: cross_val_score(p, X_tr, y_tr, cv=cv5, scoring="roc_auc") for name, p in candidates.items()}
for name, s in results.items():
    print(f"{name:>28}: CV ROC-AUC {s.mean():.3f} ± {s.std():.3f}")

fig, ax = plt.subplots(figsize=(8, 3.2))
names_ = list(results)
ax.barh(names_, [results[n].mean() for n in names_], xerr=[results[n].std() for n in names_], color=[GREY, BLUE, ORANGE], capsize=5)
ax.set(title="Churn model: 5-fold CV ROC-AUC by feature set", xlabel="ROC-AUC", xlim=(0.5, 1.0)); plt.show()
means = [results[n].mean() for n in names_]
assert means[0] < means[1] < means[2]

In [ ]:
search = GridSearchCV(build_pipeline(), {"model__C": [0.1, 1, 10], "prep__text__ngram_range": [(1, 1), (1, 2)]},
                      cv=cv5, scoring="roc_auc").fit(X_tr, y_tr)
final = search.best_estimator_
from sklearn.metrics import roc_auc_score
test_auc = roc_auc_score(y_te, final.predict_proba(X_te)[:, 1])
print("best settings:", search.best_params_, f"  CV ROC-AUC {search.best_score_:.3f}")
print(f"test ROC-AUC (the one look): {test_auc:.3f}")

coefs = pd.Series(final.named_steps["model"].coef_[0], index=final.named_steps["prep"].get_feature_names_out())
print("\nstrongest churn signals:", coefs.sort_values(ascending=False).head(5).round(2).to_dict())
print("strongest retention signals:", coefs.sort_values().head(3).round(2).to_dict())
assert test_auc > 0.75

Parameters inside a pipeline are addressed as `step__substep__param` (`prep__text__ngram_range`), so the grid search tunes preprocessing and model together, inside every fold. The top coefficients recover what we built into the data: unhappy ticket phrases ("cancelling my", and "we" / "is" from "we are switching" and "this is too expensive") and monthly contracts push towards churn; tenure and annual contracts pull away. That "is" and "we" rank so high is an artefact of our six template sentences, a reminder to read the features rather than trust them.

### A prediction function for serving

Wrap the loaded artifact in a function that takes one raw record (what an API would receive) and returns a probability. Missing fields and unseen categories are handled by the pipeline itself.

In [ ]:
def predict_churn(record: dict, artifact=loaded) -> float:
    """P(churn) for one raw customer record, using the saved pipeline."""
    row = pd.DataFrame([{col: record.get(col, np.nan) for col in artifact["features"]}])
    return float(artifact["pipeline"].predict_proba(row)[0, 1])

loyal = {"tenure_months": 60, "monthly_charges": 55.0, "support_calls": 0, "plan": "enterprise",
         "contract": "annual", "region": "north", "last_ticket": "thanks for the quick help yesterday"}
at_risk = {"tenure_months": 3, "monthly_charges": 110.0, "support_calls": 5, "plan": "basic",
           "contract": "monthly", "region": "west", "last_ticket": "this is too expensive, thinking of cancelling"}
sparse = {"tenure_months": 12, "plan": "platinum", "last_ticket": "hello"}        # missing fields + unseen category
for name, rec in [("loyal", loyal), ("at risk", at_risk), ("sparse / unseen", sparse)]:
    print(f"{name:>16}: P(churn) = {predict_churn(rec):.3f}")
assert predict_churn(at_risk) > 0.7 > 0.2 > predict_churn(loyal)

The `artifact` here is the one we saved and reloaded from disk, so this is exactly what a service would run.

## Try it yourself

**1.** Write a custom transformer: `LogTransformer` that applies `np.log1p` to the columns it is given, with `fit` / `transform` methods (inherit from `BaseEstimator, TransformerMixin`). Use it for `monthly_charges` inside a pipeline and cross-validate.

In [ ]:
# Solution — try it yourself first, then run this
class LogTransformer(BaseEstimator, TransformerMixin):
    """Stateless transformer: log1p of every input column."""
    def fit(self, X, y=None):
        return self                                     # nothing to learn, but must return self
    def transform(self, X):
        return np.log1p(np.asarray(X, dtype=float))

prep = ColumnTransformer([
    ("money", make_pipeline(SimpleImputer(strategy="median"), LogTransformer(), StandardScaler()), ["monthly_charges"]),
    ("num", StandardScaler(), ["tenure_months", "support_calls"]),
])
scores = cross_val_score(make_pipeline(prep, LogisticRegression()), X_tr, y_tr, cv=cv5, scoring="roc_auc")
print(f"CV ROC-AUC with a custom log transformer: {scores.mean():.3f}")
assert np.allclose(LogTransformer().fit_transform([[0.0], [np.e - 1]]), [[0.0], [1.0]])

**2.** Break it on purpose: build the pipeline with `OneHotEncoder(handle_unknown="error")` and predict a record with `plan="platinum"`. What error do you get, and why is `"ignore"` usually the right production choice (plus logging how often it happens)?

In [ ]:
# Solution — try it yourself first, then run this
strict = Pipeline([("prep", ColumnTransformer([
    ("cat", make_pipeline(SimpleImputer(strategy="constant", fill_value="missing"), OneHotEncoder(handle_unknown="error")), CATEGORICAL),
    ("num", SimpleImputer(strategy="median"), NUMERIC)])), ("model", LogisticRegression(max_iter=2000))]).fit(X_tr, y_tr)
try:
    strict.predict(pd.DataFrame([{**at_risk, "plan": "platinum"}]))
except ValueError as e:
    print("ValueError:", str(e)[:90], "...")

**3.** The scaler version of the leak. Standardising on all rows before cross-validating a kNN is "wrong" too. Measure how big the difference is on the churn numeric columns. Is it as dramatic as feature selection? Why not?

In [ ]:
# Solution — try it yourself first, then run this
from sklearn.neighbors import KNeighborsClassifier
Xn = SimpleImputer(strategy="median").fit_transform(X_tr[NUMERIC])
leaky = cross_val_score(KNeighborsClassifier(15), StandardScaler().fit_transform(Xn), y_tr, cv=cv5, scoring="roc_auc").mean()
clean = cross_val_score(make_pipeline(StandardScaler(), KNeighborsClassifier(15)), Xn, y_tr, cv=cv5, scoring="roc_auc").mean()
print(f"scaler outside CV {leaky:.4f}   inside CV {clean:.4f}   difference {leaky - clean:+.4f}")
# Tiny: a mean and a std per column learn almost nothing about the validation fold, and nothing about its labels.
# Feature selection learns *from the labels*, which is why it leaks so much more. Both are bugs; one is expensive.

## Say it in an interview

- **30-second answer:** "I split first, then build one `Pipeline`: a `ColumnTransformer` that imputes and scales numeric columns, imputes and one-hot encodes categoricals with `handle_unknown='ignore'`, and vectorises text with TF-IDF, followed by the model. I cross-validate and grid-search the whole pipeline, so every fitted step only ever sees training folds. I save the fitted pipeline plus library versions as one artifact and serve that, so training and serving transformations cannot drift."
- **Scalers:** Standard by default, Robust with outliers, MinMax for bounded inputs, log for long tails, none for trees.
- **Encoders:** one-hot for nominal, ordinal (with an explicit order) for ordered or for trees, target encoding (cross-fitted) for high cardinality.
- **Missing values:** median impute plus a missing-indicator; missingness is often predictive; tree boosters take NaN directly.
- **Leakage:** anything fitted (scaler, vocabulary, selector, encoder, oversampler) goes inside the pipeline inside the CV. The demo: noise data scored well above chance with selection outside CV and about 50% inside.
- **Traps:** `fit_transform` on the test set; alphabetical `OrdinalEncoder` order; `handle_unknown="error"` in production; saving the model without its preprocessing; loading a pickle across scikit-learn versions, or from an untrusted source.
- **GenAI mapping:** the prompt template, retrieval settings and model version are your "pipeline": version and ship them together (`pd23`), and keep offline and online computation identical (the feature-store idea, `dt20`).

## Next

- [ML model end to end (coding round)](../17_interview_coding_rounds/06_ml_model_end_to_end.ipynb): do this under time pressure.
- [FastAPI serving and streaming](../16_production/04_fastapi_serving_and_streaming.ipynb): put `predict_churn` behind an endpoint.
- [pandas essentials](../04_numpy_pandas_pytorch/02_pandas_essentials.ipynb) and [01 · ML workflow](01_ml_workflow_and_train_test_split.ipynb) (the four kinds of leakage).
- Theory: [ml_fundamentals course](../../ml_fundamentals/index.html) (chapters 11 and 12, and the feature-engineering tools page), [interview bank](../../ai_interview_prep/index.html).